# Notebook 6: Results Tables
**Paper 5 – Cross-Framework Quantum Algorithm Benchmarking**

Generates all LaTeX-ready tables for the paper:

| Table | Content |
|-------|--------|
| Table 1 | Mean gate count per algorithm per framework |
| Table 2 | Circuit depth comparison |
| Table 3 | Compilation time (mean ± std ms) |
| Table 4 | Simulation time (mean ± std ms, @ 4096 shots) |
| Table 5 | Pairwise JSD with equivalence labels |
| Table 6 | Quantum Volume estimates |
| **Table 7** | **QPack composite scores (NEW)** |

**Prerequisites**: Notebooks 1–4 must be complete.

**Outputs**: `benchmarks/results/tables/*.tex`

In [ ]:
import os, sys
QCANVAS_ROOT = os.path.abspath('../..')
if QCANVAS_ROOT not in sys.path:
    sys.path.insert(0, QCANVAS_ROOT)
import pandas as pd
import numpy as np

TABLES_DIR = 'benchmarks/results/tables'
os.makedirs(TABLES_DIR, exist_ok=True)

def save_latex(df, filename, caption, label, float_format='%.3f'):
    """Save a DataFrame as a complete LaTeX table."""
    latex = df.to_latex(float_format=float_format, escape=False,
                        caption=caption, label=label)
    path = os.path.join(TABLES_DIR, filename)
    with open(path, 'w') as f:
        f.write('% Auto-generated by nb06_results_tables.ipynb\n')
        f.write(f'% Paper 5 — Cross-Framework Quantum Algorithm Benchmarking\n\n')
        f.write(latex)
    print(f'Saved: {path}')

print('LaTeX table generator ready.')

In [ ]:
# Load all data
df_struct = pd.read_csv('benchmarks/metrics/structural_metrics.csv')
df_ctimes = pd.read_csv('benchmarks/metrics/compilation_times.csv')
df_sim    = pd.read_csv('benchmarks/metrics/simulation_metrics.csv')
df_qv     = pd.read_csv('benchmarks/metrics/quantum_volume_estimates.csv')
df_tests  = pd.read_csv('benchmarks/metrics/statistical_tests.csv')
df_qpack  = pd.read_csv('benchmarks/metrics/qpack_scores.csv')

print('All metrics CSV files loaded.')

In [ ]:
# ── Table 1: Gate count per algorithm per framework ──────────────────────────
df_fixed = df_struct[df_struct['n_qubits'] == df_struct.groupby('algorithm')['n_qubits'].transform('min')]
t1 = df_fixed.pivot_table(index='algorithm', columns='framework', values='total_gates').round(0).astype(int)
t1.columns.name = 'Framework'
t1 = t1.rename(columns={'qiskit': 'Qiskit', 'cirq': 'Cirq', 'pennylane': 'PennyLane'})
print('Table 1 — Gate Count:')
display(t1)
save_latex(t1, 'table1_gate_counts.tex',
           caption='Total gate count per algorithm and framework (compiled to OpenQASM 3.0 via QCanvas, minimum qubit count shown).',
           label='tab:gate-counts', float_format='%d')

In [ ]:
# ── Table 2: Circuit depth ────────────────────────────────────────────────────
t2 = df_fixed.pivot_table(index='algorithm', columns='framework', values='circuit_depth').round(0).astype(int)
t2 = t2.rename(columns={'qiskit': 'Qiskit', 'cirq': 'Cirq', 'pennylane': 'PennyLane'})
print('Table 2 — Circuit Depth:')
display(t2)
save_latex(t2, 'table2_circuit_depth.tex',
           caption='Estimated circuit depth per algorithm and framework.',
           label='tab:circuit-depth', float_format='%d')

In [ ]:
# ── Table 3: Compilation time (mean ± std) ───────────────────────────────────
df_ct_success = df_ctimes[df_ctimes['success']]
ct_agg = df_ct_success.groupby(['algorithm', 'framework'])[['mean_compile_ms', 'std_compile_ms']].mean()
ct_agg['compile_ms_str'] = ct_agg.apply(
    lambda r: f"{r['mean_compile_ms']:.2f} ± {r['std_compile_ms']:.2f}", axis=1
)
t3 = ct_agg['compile_ms_str'].unstack('framework')
t3 = t3.rename(columns={'qiskit': 'Qiskit', 'cirq': 'Cirq', 'pennylane': 'PennyLane'})
print('Table 3 — Compilation Time (mean ± std ms):')
display(t3)
save_latex(t3, 'table3_compilation_time.tex',
           caption='Compilation time (mean $\pm$ std, ms, $N=10$ repeats) per algorithm and framework.',
           label='tab:compile-time')

In [ ]:
# ── Table 4: Simulation time (4096 shots) ────────────────────────────────────
df_4k = df_sim[(df_sim['shots'] == 4096) & df_sim['success']]
df_4k_fix = df_4k[df_4k['n_qubits'] == df_4k.groupby('algorithm')['n_qubits'].transform('min')]
sim_agg = df_4k_fix.groupby(['algorithm', 'framework'])[['mean_sim_time_ms', 'std_sim_time_ms']].mean()
sim_agg['sim_ms_str'] = sim_agg.apply(
    lambda r: f"{r['mean_sim_time_ms']:.1f} ± {r['std_sim_time_ms']:.1f}", axis=1
)
t4 = sim_agg['sim_ms_str'].unstack('framework')
t4 = t4.rename(columns={'qiskit': 'Qiskit', 'cirq': 'Cirq', 'pennylane': 'PennyLane'})
print('Table 4 — Simulation Time (4096 shots, mean ± std ms):')
display(t4)
save_latex(t4, 'table4_simulation_time.tex',
           caption='Simulation time (mean $\pm$ std, ms) at 4096 shots per algorithm and framework. All frameworks simulated on the same QSim Cirq backend.',
           label='tab:sim-time')

In [ ]:
# ── Table 5: Pairwise JSD with labels ────────────────────────────────────────
t5_data = df_tests[['algorithm', 'n_qubits', 'jsd_qiskit_cirq', 'jsd_qiskit_pl',
                      'jsd_cirq_pl', 'label_qk_cq', 'label_qk_pl', 'label_cq_pl']].copy()

# Combine JSD value and label
for pair, label_col in [('jsd_qiskit_cirq', 'label_qk_cq'),
                          ('jsd_qiskit_pl',   'label_qk_pl'),
                          ('jsd_cirq_pl',     'label_cq_pl')]:
    col_name = pair.replace('jsd_', '').replace('_', '↔')
    t5_data[col_name] = t5_data.apply(
        lambda r: f"{r[pair]:.4f} {r[label_col]}" if pd.notna(r[pair]) else '-', axis=1
    )

t5 = t5_data.set_index('algorithm')[['qiskit↔cirq', 'qiskit↔pl', 'cirq↔pl']]
t5.columns = ['Qiskit↔Cirq', 'Qiskit↔PennyLane', 'Cirq↔PennyLane']
print('Table 5 — Pairwise JSD (✓ < 0.01, ~ < 0.05, ✗ ≥ 0.05):')
display(t5)
save_latex(t5, 'table5_pairwise_jsd.tex',
           caption='Pairwise Jensen-Shannon Divergence (JSD, $\sqrt{}$ form) across all algorithms at 4096 shots. \\checkmark: JSD $<$ 0.01 (equivalent); $\\sim$: marginal; \\xmark: JSD $\\geq$ 0.05 (divergent).',
           label='tab:jsd')

In [ ]:
# ── Table 6: Quantum Volume estimates ────────────────────────────────────────
df_qv_fix = df_qv[df_qv['n_qubits'] == df_qv.groupby('algorithm')['n_qubits'].transform('min')]
t6 = df_qv_fix.pivot_table(index='algorithm', columns='framework',
                              values='effective_qv').round(0).astype(int)
t6 = t6.rename(columns={'qiskit': 'Qiskit', 'cirq': 'Cirq', 'pennylane': 'PennyLane'})
print('Table 6 — Quantum Volume Estimates:')
display(t6)
save_latex(t6, 'table6_quantum_volume.tex',
           caption='Estimated Quantum Volume per algorithm and framework. Note: estimated values based on depolarizing + T1 noise model; not measured on real hardware.',
           label='tab:quantum-volume', float_format='%d')

In [ ]:
# ── Table 7: QPack Composite Scores (NEW) ────────────────────────────────────
if not df_qpack.empty:
    df_qpack_display = df_qpack.set_index('framework').rename(
        index={'qiskit': 'Qiskit', 'cirq': 'Cirq', 'pennylane': 'PennyLane'}
    ).rename(columns={
        'S_compile_speed': '$S_{\\text{speed}}$',
        'S_scalability':   '$S_{\\text{scale}}$',
        'S_accuracy':      '$S_{\\text{acc}}$',
        'S_capacity':      '$S_{\\text{cap}}$',
        'S_overall':       '$S_{\\text{overall}}$',
    })
    print('Table 7 — QPack Composite Scores:')
    display(df_qpack_display.round(2))

    # Add QPack reference row
    ref_note = "QPack ref (QuEST=183.2, Cirq=157.6, Qiskit Aer=147.2)."

    save_latex(df_qpack_display.round(2), 'table7_qpack_scores.tex',
               caption=f'QPack-adapted composite benchmark scores per framework. {ref_note}',
               label='tab:qpack-scores')
else:
    print('[SKIP] qpack_scores.csv not found — run nb03 first.')